# OmniDiag — Heart: Hyperparameter Search & Threshold SelectionTwo questions, answered with evidence rather than assertion:**1. Which hyperparameters, and how much did tuning actually buy?**Tuning and then reporting the same cross-validation that guided the tuning iscircular — the score absorbs the search. This notebook measures that inflationinstead of hiding it, by running the whole search *inside* each leave-one-site-outfold (nested), so the held-out hospital never touches the search.**2. Which decision threshold, and why that rule?**Seven selection rules are implemented. Each is applied to out-of-fold predictionsof the training data only, then judged on held-out sites and on the externalTehran cohort. The table at the end is the justification for the threshold thatships.**Two deliberate choices you should be ready to defend**- The search objective is **ROC-AUC**, not accuracy. `configs/heart_disease.yaml`  currently declares `optimization_metric: "accuracy"`. On a cohort where  prevalence swings from 36% to 94% between hospitals, accuracy rewards  predicting the majority class. AUC is threshold-free and prevalence-robust;  the operating point is then chosen separately in Part B.- Threshold selection uses an explicit clinical cost, not F1. F1 treats a missed  patient and a false alarm as equally bad. For triage they are not.**Inputs:** `uci_heart_by_site.csv`, `z_alizadeh_translated.csv`

In [ ]:
!pip -q install optuna xgboost --upgrade 2>/dev/nullimport json, os, glob, warnings, timewarnings.filterwarnings("ignore")import numpy as np, pandas as pd, xgboost as xgb, optuna, joblibimport matplotlib; matplotlib.use("Agg")import matplotlib.pyplot as pltoptuna.logging.set_verbosity(optuna.logging.WARNING)from sklearn.compose import ColumnTransformerfrom sklearn.experimental import enable_iterative_imputer  # noqafrom sklearn.impute import IterativeImputer, SimpleImputerfrom sklearn.metrics import confusion_matrix, roc_auc_score, roc_curvefrom sklearn.model_selection import StratifiedKFold, cross_val_predictfrom sklearn.pipeline import Pipelinefrom sklearn.preprocessing import OrdinalEncoder, StandardScalerSEED = 42N_TRIALS = 60          # raise if you have time; 60 is enough to plateau hereFN_COST, FP_COST = 2.0, 1.0MIN_SENS = 0.90        # for the sensitivity-constrained rulerng = np.random.default_rng(SEED)for d in ["models", "evidence", "figures"]:    os.makedirs(d, exist_ok=True)def find(n):    h = glob.glob(f"/kaggle/input/**/{n}", recursive=True) or glob.glob(n)    if not h: raise FileNotFoundError(n)    return h[0]uci = pd.read_csv(find("uci_heart_by_site.csv"))zal = pd.read_csv(find("z_alizadeh_translated.csv"))NUM = ["Age","RestingBP","Cholesterol","MaxHR","Oldpeak","FastingBS"]CAT = ["Sex","ChestPainType","RestingECG","ExerciseAngina","ST_Slope"]NUM_R, CAT_R = ["Age","RestingBP","Cholesterol","FastingBS"], ["Sex","ChestPainType","RestingECG"]FULL, RED, TGT = NUM+CAT, NUM_R+CAT_R, "HeartDisease"X, y, site = uci[FULL], uci[TGT], uci["site"]cv = StratifiedKFold(5, shuffle=True, random_state=SEED)CONFIG_PARAMS = dict(n_estimators=898, max_depth=5, learning_rate=0.013594126498405943,                     subsample=0.963733407970185, colsample_bytree=0.5454718650965412,                     random_state=SEED, eval_metric="logloss")DEFAULT_PARAMS = dict(random_state=SEED, eval_metric="logloss")def make_pipe(params, num=NUM, cat=CAT):    return Pipeline([        ("prep", ColumnTransformer([            ("num", Pipeline([("imp", IterativeImputer(random_state=SEED, max_iter=10)),                              ("sc", StandardScaler())]), num),            ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),                              ("enc", OrdinalEncoder(handle_unknown="use_encoded_value",                                                     unknown_value=-1))]), cat)])),        ("clf", xgb.XGBClassifier(**params))])print(f"{len(uci)} UCI patients / {uci.site.nunique()} sites  |  {len(zal)} Tehran")

## Part A · 1 — The search spaceRanges are wide enough to include the current configuration, so the search canreject it or reproduce it. `scale_pos_weight` is included because prevalencediffers sharply between hospitals.

In [ ]:
def suggest(trial):    return dict(        n_estimators     = trial.suggest_int("n_estimators", 200, 1500, step=50),        max_depth        = trial.suggest_int("max_depth", 2, 8),        learning_rate    = trial.suggest_float("learning_rate", 5e-3, 0.3, log=True),        subsample        = trial.suggest_float("subsample", 0.5, 1.0),        colsample_bytree = trial.suggest_float("colsample_bytree", 0.4, 1.0),        min_child_weight = trial.suggest_int("min_child_weight", 1, 12),        gamma            = trial.suggest_float("gamma", 0.0, 5.0),        reg_alpha        = trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),        reg_lambda       = trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),        scale_pos_weight = trial.suggest_float("scale_pos_weight", 0.5, 3.0),        random_state=SEED, eval_metric="logloss")def search(Xs, ys, n_trials=N_TRIALS, inner=None, seed=SEED):    inner = inner or StratifiedKFold(5, shuffle=True, random_state=seed)    def obj(trial):        p = suggest(trial)        oof = cross_val_predict(make_pipe(p), Xs, ys, cv=inner, method="predict_proba")[:,1]        return roc_auc_score(ys, oof)    st = optuna.create_study(direction="maximize",                             sampler=optuna.samplers.TPESampler(seed=seed))    st.optimize(obj, n_trials=n_trials, show_progress_bar=False)    return st

## Part A · 2 — What tuning buys, and what it only appears to buy`pooled CV` is the score the search itself maximised, so it is contaminated bydefinition. `LOSO` retrains from scratch per fold. The distance between the twocolumns is the honest cost of believing a tuned CV score.

In [ ]:
t0 = time.time()study = search(X, y)BEST = {**study.best_params, "random_state": SEED, "eval_metric": "logloss"}print(f"{N_TRIALS} trials in {time.time()-t0:.0f}s   best inner CV AUC {study.best_value:.4f}\n")for k, v in study.best_params.items():    print(f"  {k:<18}{v}")def pooled_auc(params):    oof = cross_val_predict(make_pipe(params), X, y, cv=cv, method="predict_proba")[:,1]    return roc_auc_score(y, oof), oofdef loso_auc(params, tune_inside=False):    aucs, probs = {}, {}    for s in uci.site.unique():        tr, te = site != s, site == s        p = search(X[tr], y[tr], n_trials=N_TRIALS).best_params if tune_inside else params        if tune_inside: p = {**p, "random_state": SEED, "eval_metric": "logloss"}        m = make_pipe(p).fit(X[tr], y[tr])        pr = m.predict_proba(X[te])[:,1]        aucs[s], probs[s] = roc_auc_score(y[te], pr), pr    return aucs, probsrows = []cand = {"xgboost defaults": DEFAULT_PARAMS,        "current config (898/5/0.0136)": CONFIG_PARAMS,        "Optuna best": BEST}oofs = {}for name, p in cand.items():    pa, oo = pooled_auc(p); oofs[name] = oo    la, _ = loso_auc(p)    rows.append(dict(model=name, pooled_cv=round(pa,4),                     loso_mean=round(float(np.mean(list(la.values()))),4),                     loso_min=round(min(la.values()),4),                     optimism=round(pa-float(np.mean(list(la.values()))),4)))res = pd.DataFrame(rows)display(res)

In [ ]:
# Nested: the search is re-run inside every fold, so no site informs its own scoret0 = time.time()nested_auc, nested_probs = loso_auc(None, tune_inside=True)print(f"nested search finished in {time.time()-t0:.0f}s\n")nested_mean = float(np.mean(list(nested_auc.values())))for s, a in nested_auc.items():    print(f"  {s:<16}{a:.4f}")print(f"\n  nested LOSO mean           {nested_mean:.4f}")print(f"  non-nested LOSO (tuned once) {res.loc[res.model=='Optuna best','loso_mean'].iloc[0]:.4f}")print(f"  tuned pooled CV              {res.loc[res.model=='Optuna best','pooled_cv'].iloc[0]:.4f}")print(f"\n  >>> report {nested_mean:.4f} as the tuned generalisation estimate.")json.dump(dict(n_trials=N_TRIALS, objective="mean ROC-AUC, inner 5-fold CV",               best_params=study.best_params,               best_inner_cv_auc=round(study.best_value,4),               comparison=res.to_dict("records"),               nested_loso={k: round(v,4) for k,v in nested_auc.items()},               nested_loso_mean=round(nested_mean,4),               note="Nested = hyperparameter search re-run inside each LOSO fold. "                    "This is the only estimate where the held-out hospital was "                    "never involved in choosing the hyperparameters."),          open("evidence/hpo.json","w"), indent=2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,3.6), dpi=140)h = [t.value for t in study.trials if t.value is not None]axes[0].plot(h, ".", ms=4, alpha=.5)axes[0].plot(np.maximum.accumulate(h), lw=1.8, color="crimson", label="running best")axes[0].set_xlabel("trial"); axes[0].set_ylabel("inner CV ROC-AUC")axes[0].set_title(f"Optuna search ({N_TRIALS} trials)"); axes[0].legend(fontsize=7.5)w = 0.35; idx = np.arange(len(res))axes[1].bar(idx-w/2, res.pooled_cv, w, label="pooled CV (contaminated)", color="#E45756")axes[1].bar(idx+w/2, res.loso_mean, w, label="LOSO mean (honest)", color="#4C78A8")axes[1].axhline(nested_mean, ls="--", lw=1.2, color="k", label=f"nested {nested_mean:.3f}")axes[1].set_xticks(idx); axes[1].set_xticklabels(res.model, fontsize=6.5, rotation=12)axes[1].set_ylim(0.6,1.0); axes[1].set_ylabel("ROC-AUC")axes[1].set_title("What tuning buys vs what it appears to buy")axes[1].legend(fontsize=7)fig.tight_layout(); fig.savefig("figures/hpo_summary.png"); plt.close(fig)imp = optuna.importance.get_param_importances(study)fig, ax = plt.subplots(figsize=(5,3.2), dpi=140)ax.barh(list(imp)[::-1], list(imp.values())[::-1], color="#4C78A8")ax.set_title("Hyperparameter importance"); ax.set_xlabel("relative importance")fig.tight_layout(); fig.savefig("figures/hpo_importance.png"); plt.close(fig)display(pd.Series(imp).round(4).to_frame("importance"))

## Part B · 1 — Seven threshold rulesEvery rule sees only out-of-fold predictions from the **training** data. Theheld-out site and the Tehran cohort are scored afterwards, never consulted.

In [ ]:
def conf(y_, p_, t):    return confusion_matrix(y_, (p_ >= t).astype(int), labels=[0,1]).ravel()GRID = np.linspace(0.01, 0.99, 200)def r_default(y_, p_):   return 0.5def r_youden(y_, p_):    f, t, th = roc_curve(y_, p_); return float(th[np.argmax(t-f)])def r_f1(y_, p_):    best, bs = .5, -1    for t in GRID:        tn,fp,fn,tp = conf(y_,p_,t)        s = 2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else 0        if s > bs: bs, best = s, float(t)    return bestdef _cost(mult):    def f(y_, p_):        best, bc = .5, np.inf        for t in GRID:            tn,fp,fn,tp = conf(y_,p_,t)            c = mult*fn + FP_COST*fp            if c < bc: bc, best = c, float(t)        return best    return fdef r_sens_floor(y_, p_):    ok = []    for t in GRID:        tn,fp,fn,tp = conf(y_,p_,t)        se = tp/(tp+fn) if tp+fn else 0        sp = tn/(tn+fp) if tn+fp else 0        if se >= MIN_SENS: ok.append((sp, float(t)))    return max(ok)[1] if ok else 0.5def r_prevalence(y_, p_):    return float(np.quantile(p_, 1-y_.mean()))RULES = { "default 0.50":              (r_default,    "no selection; what the frontend shows today"), "Youden J (max sens+spec)":  (r_youden,     "symmetric: a miss and a false alarm cost the same"), "max F1":                    (r_f1,         "favours precision; ignores clinical asymmetry"), "cost FN x2  (chosen)":      (_cost(2.0),   "a missed patient costs twice a false alarm"), "cost FN x3":                (_cost(3.0),   "more aggressive triage"), f"sensitivity >= {MIN_SENS:.0%}":(r_sens_floor,"fix the miss rate, maximise specificity under it"), "prevalence matched":        (r_prevalence, "predict positives at the observed base rate"),}print(f"{len(RULES)} rules on a {len(GRID)}-point grid over [0.01, 0.99]")

## Part B · 2 — Each rule judged on data it did not choose itself

In [ ]:
PROD = BEST if res.loc[res.model=="Optuna best","loso_mean"].iloc[0] >= \              res.loc[res.model=="current config (898/5/0.0136)","loso_mean"].iloc[0] else CONFIG_PARAMSprint("hyperparameters carried into Part B:",      "Optuna best" if PROD is BEST else "current config", "\n")oof_prod = cross_val_predict(make_pipe(PROD), X, y, cv=cv, method="predict_proba")[:,1]# held-out probabilities per site, using the production hyperparameterssite_probs, site_thr = {}, {}for s in uci.site.unique():    tr, te = site != s, site == s    oof_tr = cross_val_predict(make_pipe(PROD), X[tr], y[tr], cv=cv, method="predict_proba")[:,1]    site_thr[s] = (y[tr].values, oof_tr)    site_probs[s] = (y[te].values,                     make_pipe(PROD).fit(X[tr], y[tr]).predict_proba(X[te])[:,1])red_pipe = make_pipe(PROD, NUM_R, CAT_R).fit(uci[RED], y)oof_red  = cross_val_predict(make_pipe(PROD, NUM_R, CAT_R), uci[RED], y, cv=cv,                             method="predict_proba")[:,1]teh = (zal[TGT].values, red_pipe.predict_proba(zal[RED])[:,1])def score(y_, p_, t):    tn,fp,fn,tp = conf(y_,p_,t)    return dict(sens=tp/(tp+fn) if tp+fn else np.nan,                spec=tn/(tn+fp) if tn+fp else np.nan,                fn=int(fn), fp=int(fp), cost=FN_COST*fn+FP_COST*fp,                acc=(tp+tn)/len(y_))rows = []for name,(rule,_) in RULES.items():    ths, se, sp, fn_, fp_, cost = [], [], [], [], [], []    for s,(ytr,otr) in site_thr.items():        t = rule(ytr, otr); ths.append(t)        m = score(*site_probs[s], t)        se.append(m["sens"]); sp.append(m["spec"])        fn_.append(m["fn"]); fp_.append(m["fp"]); cost.append(m["cost"])    t_teh = rule(y.values, oof_red)    mt = score(*teh, t_teh)    rows.append(dict(rule=name,                     thr_mean=round(float(np.mean(ths)),3),                     thr_spread=round(float(max(ths)-min(ths)),3),                     loso_sens=round(float(np.mean(se))*100,1),                     loso_spec=round(float(np.mean(sp))*100,1),                     loso_FN=int(sum(fn_)), loso_FP=int(sum(fp_)),                     loso_cost=int(sum(cost)),                     tehran_sens=round(mt["sens"]*100,1),                     tehran_spec=round(mt["spec"]*100,1),                     tehran_FN=mt["fn"]))thr_table = pd.DataFrame(rows).sort_values("loso_cost")display(thr_table)print("\nloso_FN / loso_FP are totals across all four held-out hospitals (920 patients).")

## Part B · 3 — Reading the table`loso_cost` is the clinical objective made explicit, so the rule minimising it isthe rule that best serves triage. Three things to check before accepting it:- **`thr_spread`** — how far the chosen number moves when the training sites  change. A large spread means the threshold is a property of the cohort, not of  the model, and must be recalibrated per deployment site.- **`loso_FN`** — patients sent home with disease. This is the number a  cardiologist will react to.- **`tehran_*`** — whether the rule survives a cohort from another country.

In [ ]:
best_rule = thr_table.iloc[0]chosen = thr_table[thr_table.rule.str.contains("chosen")].iloc[0]print(f"lowest clinical cost : {best_rule['rule']}  (cost {best_rule.loso_cost}, "      f"FN {best_rule.loso_FN}, sens {best_rule.loso_sens}%)")print(f"rule shipped         : {chosen['rule']}  (cost {chosen.loso_cost}, "      f"FN {chosen.loso_FN}, sens {chosen.loso_sens}%)")print(f"default 0.50 costs   : "      f"{int(thr_table[thr_table.rule=='default 0.50'].loso_cost.iloc[0])} "      f"with {int(thr_table[thr_table.rule=='default 0.50'].loso_FN.iloc[0])} missed patients")print(f"\nthreshold instability across training sets: "      f"{chosen.thr_spread:.3f} absolute spread "      f"-> a fixed shipped threshold needs per-site recalibration.")fig, axes = plt.subplots(1, 2, figsize=(11,3.8), dpi=140)t = thr_table.set_index("rule")t[["loso_FN","loso_FP"]].plot(kind="barh", ax=axes[0], color=["#E45756","#F2CF5B"])axes[0].set_title("Missed patients vs false alarms (sum over held-out sites)")axes[0].set_xlabel("patients"); axes[0].tick_params(labelsize=7)axes[1].scatter(100-t.loso_spec, t.loso_sens, s=60, color="#4C78A8")for r in t.index:    axes[1].annotate(r, (100-t.loc[r,"loso_spec"], t.loso_sens[r]), fontsize=6.5,                     xytext=(3,3), textcoords="offset points")axes[1].set_xlabel("1 - specificity (%)"); axes[1].set_ylabel("sensitivity (%)")axes[1].set_title("Where each rule lands, on held-out sites")fig.tight_layout(); fig.savefig("figures/threshold_rules.png"); plt.close(fig)thr_table.to_json("evidence/threshold_rules.json", orient="records", indent=2)json.dump({k: v[1] for k, v in RULES.items()},          open("evidence/threshold_rule_rationale.json","w"), indent=2)

## Part C — Ship it

In [ ]:
FINAL_THR = RULES["cost FN x2  (chosen)"][0](y.values, oof_prod)FINAL_THR_RED = RULES["cost FN x2  (chosen)"][0](y.values, oof_red)full_model = make_pipe(PROD).fit(X, y)joblib.dump(dict(pipeline=full_model, features=FULL, numeric=NUM, categorical=CAT,                 threshold=round(FINAL_THR,4), params=PROD,                 selection_rule="cost = 2.0*FN + 1.0*FP on out-of-fold training predictions",                 trained_on="UCI 4 sites", n=int(len(y))), "models/heart_full_tuned.pkl")joblib.dump(dict(pipeline=red_pipe, features=RED, threshold=round(FINAL_THR_RED,4),                 params=PROD, trained_on="UCI 4 sites (shared features)"),            "models/heart_reduced_tuned.pkl")summary = dict(  hyperparameters=dict(source="Optuna" if PROD is BEST else "existing config",                       params={k:v for k,v in PROD.items() if k!="eval_metric"},                       objective="ROC-AUC (not accuracy)",                       honest_estimate_nested_loso=round(nested_mean,4),                       contaminated_pooled_cv=round(                         float(res.loc[res.model=="Optuna best","pooled_cv"].iloc[0]),4)),  threshold=dict(shipped=round(FINAL_THR,4), shipped_reduced=round(FINAL_THR_RED,4),                 rule="cost = 2.0*FN + 1.0*FP",                 selected_on="out-of-fold predictions of training data",                 rules_compared=thr_table.to_dict("records"),                 instability_across_training_sets=float(chosen.thr_spread),                 caveat="Threshold is cohort-dependent; recalibrate per deployment site."),  files=sorted(glob.glob("evidence/*")+glob.glob("figures/*")+glob.glob("models/*")))json.dump(summary, open("evidence/hpo_threshold_summary.json","w"), indent=2)print(json.dumps(summary["hyperparameters"], indent=2))print(json.dumps({k:v for k,v in summary["threshold"].items() if k!="rules_compared"}, indent=2))

In [ ]:
!cd /kaggle/working && zip -qr omnidiag_hpo_thresholds.zip models evidence figures && ls -la omnidiag_hpo_thresholds.zip